# Calculate and plot power versus radial uv distance
## For making Figures 14 and 15 in the paper
### A. Ordog, Sept 3, 2024
### based on 'COMBINE_RADIAL_ANALYSIS' series of notebooks from PhD

In [ ]:
import astropy.io.fits as pf
from astropy.io import fits
from astropy.wcs import WCS
import numpy as np
import matplotlib.pyplot as plt
import math
from mpl_toolkits.axes_grid1 import make_axes_locatable
from matplotlib.colors import LogNorm
import os
import subprocess
import importlib as imp
import copy
import scipy.stats
from matplotlib.pyplot import cm
from scipy.stats import kde
#import figure_subroutines as subs
from astropy import units as u
from astropy.coordinates import SkyCoord
from pylab import *
from matplotlib import ticker

In [ ]:
def get_fields_and_mosaics(mos_file):
    
    mos_fields = {}
    fields = []
    
    with open(mos_file, 'r') as file:
        for line in file:
            if len(line.strip().split()) == 2:
                mos = line.strip().split()[0]
                mos_fields[mos] = []
            if len(line.strip().split()) == 1:
                field = line.strip().split()[0]
                mos_fields[mos].append(field)
                fields.append(field)
                
    print('Number of mosaics: ',len(mos_fields))
    print('Number of fields in mosaics: ',len(fields))
    
    fields = set(fields)
    print('Number of unique fields: ',len(fields))
    
    return mos_fields, fields

In [ ]:
print('Original CGPS')
mos_fields_old, fieldnames_old = get_fields_and_mosaics('/home/ordoga/DRAO_export/CG_W23/mos_plots/mosaics_fields_list.txt')
print()
print('Actual OB + CGPS')
mos_fields, fieldnames = get_fields_and_mosaics('/home/ordoga/DRAO_export/mosaics_fields_list_OB_and_CGPS.txt')


In [ ]:
diff = list(set(fieldnames) - set(fieldnames_old))

print(diff)

# 384 CGPS
# 1 extra CGPS
# 14 OB (not OB06,OB07,OB01)
# 4 extra archival
print(384+1+14+4)

## Make dictionary for all fields

In [ ]:
#fieldnames = ['ob02','ob03','ob04','ob05',
#              'ob08','ob09','ob10','ob11',
#              'ob12','ob13','ob14','ob15',
#              'ob16','ob17','sn01','sn02',
#              'sn03','rr22']
#fieldnames = ['ej1','ob03','ob04']

fieldnames = list(fieldnames)

numfields = len(fieldnames)
print(numfields)
rmax = 200

data = {}
#channels = {'a':np.zeros(rmax),'b':np.zeros(rmax),'c':np.zeros(rmax),'d':np.zeros(rmax)}
#cgps  = {'raw':channels.copy(), 'feather':channels.copy()}
#gmims = {'raw':channels.copy(),  'deconv':channels.copy(), 'feather':channels.copy()}

for field in fieldnames:
    data[field] = {'stokesq': {'cgps':{'raw':{'a':np.zeros(rmax),'b':np.zeros(rmax),
                                              'c':np.zeros(rmax),'d':np.zeros(rmax)}, 
                                       'feather':{'a':np.zeros(rmax),'b':np.zeros(rmax),
                                                  'c':np.zeros(rmax),'d':np.zeros(rmax)}},
                               'gmims':{'raw':{'a':np.zeros(rmax),'b':np.zeros(rmax),
                                               'c':np.zeros(rmax),'d':np.zeros(rmax)},  
                                        'deconv':{'a':np.zeros(rmax),'b':np.zeros(rmax),
                                                  'c':np.zeros(rmax),'d':np.zeros(rmax)}, 
                                        'feather':{'a':np.zeros(rmax),'b':np.zeros(rmax),
                                                   'c':np.zeros(rmax),'d':np.zeros(rmax)}},
                               'ratio':{'a':np.nan,'b':np.nan,'c':np.nan,'d':np.nan}},
                   'stokesu': {'cgps':{'raw':{'a':np.zeros(rmax),'b':np.zeros(rmax),
                                              'c':np.zeros(rmax),'d':np.zeros(rmax)}, 
                                       'feather':{'a':np.zeros(rmax),'b':np.zeros(rmax),
                                                  'c':np.zeros(rmax),'d':np.zeros(rmax)}},
                               'gmims':{'raw':{'a':np.zeros(rmax),'b':np.zeros(rmax),
                                               'c':np.zeros(rmax),'d':np.zeros(rmax)},  
                                        'deconv':{'a':np.zeros(rmax),'b':np.zeros(rmax),
                                                  'c':np.zeros(rmax),'d':np.zeros(rmax)}, 
                                        'feather':{'a':np.zeros(rmax),'b':np.zeros(rmax),
                                                   'c':np.zeros(rmax),'d':np.zeros(rmax)}},
                               'ratio':{'a':np.nan,'b':np.nan,'c':np.nan,'d':np.nan}},
                   'uvr2d':{'a':np.zeros(rmax),'b':np.zeros(rmax),
                          'c':np.zeros(rmax),'d':np.zeros(rmax)},
                   'uvr1d':{'a':np.zeros(rmax),'b':np.zeros(rmax),
                          'c':np.zeros(rmax),'d':np.zeros(rmax)}}
    

In [ ]:
def read_data(file):
    
    hdu = fits.open(file)
    hdr = hdu[0].header
    
    n1 = int(hdr['NAXIS1']/2)
    n2 = int(hdr['NAXIS2']-1)
    #print(n1,n2)
    
    data = hdu[0].data[0][0][1:n2+1,0:n1]
    #print(data.shape)


    return data, hdr

In [ ]:
def make_axis_lists_2D(hdr):
    
    nx = hdr['NAXIS1']
    ny = hdr['NAXIS2']
   
    dx = hdr['CDELT1']
    dy = hdr['CDELT2']
    
    xpix = hdr['CRPIX1']
    ypix = hdr['CRPIX2']
    
    xval = hdr['CRVAL1']
    yval = hdr['CRVAL2']
    
    x = np.arange(nx)+1-xpix
    y = np.arange(ny)+1-ypix

    lon_ax = x*dx+xval
    lat_ax = y*dy+yval
    
    return lon_ax, lat_ax

In [ ]:
def get_uv_radii(hdr):

    n1 = int(hdr['NAXIS1']/2)
    n2 = int(hdr['NAXIS2']-1)
    
    ruv = np.zeros([n2,n1])
    
    u,v = make_axis_lists_2D(hdr)
    
    for i in range(0,n1):
        for j in range(0,n2):
            ruv[j,i] = np.sqrt((np.flip(u[1:n1+1])[i])**2 + v[1:n2+1][j]**2)
        
    return ruv

In [ ]:
def plot_uv_radius2D(uvr_field):
    
    
    fig,axs = plt.subplots(1,4,figsize=(12,5))
    for i in range(0,4):
        axs[i].imshow(uvr_field[i],vmin=0,vmax=1000,cmap='gray_r')
        
    fig,ax = plt.subplots(1,1,figsize=(12,5))
    for i in range(0,4):
        idx = int(floor(uvr_field[i].shape[0]/2))    
        ax.plot(uvr_field[i][idx,:])
        
    return
        

In [ ]:
def binning(uvmap,uvr,rmax,r):
    
    binned = np.empty_like(r)
    
    for i in range(0,rmax):
    
        wx = np.where((uvr == r[i]))[1]
        wy = np.where((uvr == r[i]))[0]
        binned[i] = np.nanmean(uvmap[wy,wx])
    
    return binned

In [ ]:
def make_radial_plots(data,field,stokes,band):
    
    fs=22
    lw = 2

    axs = ['ax1','ax2','ax3']

    fig = plt.figure(figsize=(17,15))
    plt.subplots_adjust(top = 0.98, bottom = 0.06, right = 0.98, left = 0.08, hspace=0.1)

    axs[0] = fig.add_subplot(311)
    #axs[0].scatter(data[field]['uvr1d2d'][band],ampl_C_initial[j],s=60,label='ST: original visibilities')
    #axs[0].scatter(data[field]['uvr1d2d'][band],ampl_G_initial[j],s=50,label='SA: visibilities after initial taper')
    axs[0].scatter(data[field]['uvr1d'][band],data[field][stokes]['cgps']['raw'][band],color='blue',label='ST: mean')
    axs[0].scatter(data[field]['uvr1d'][band],data[field][stokes]['gmims']['raw'][band],color='red',label='SA: mean')
    handles,labels = axs[0].get_legend_handles_labels()
    order = [2,0,3,1]
    axs[0].set_ylim(0,4000)
    #axs[0].legend([handles[idx] for idx in order], [labels[idx] for idx in order],fontsize=fs)

    axs[1] = fig.add_subplot(312)
    #axs[1].scatter(data[field]['uvr1d2d'][band],ampl_C_initial[j],s=60,label='ST: original visibilities')
    #axs[1].scatter(data[field]['uvr1d2d'][band],ampl_G_match[j],s=50,label='SA: visibilities matched to ST beam')
    axs[1].scatter(data[field]['uvr1d'][band],data[field][stokes]['cgps']['raw'][band],color='blue',label='ST: mean')
    axs[1].scatter(data[field]['uvr1d'][band],data[field][stokes]['gmims']['deconv'][band],color='red',label='SA: mean')
    axs[1].set_ylim(0,4000)
    handles,labels = axs[1].get_legend_handles_labels()
    order = [2,0,3,1]
    #axs[1].legend([handles[idx] for idx in order], [labels[idx] for idx in order],fontsize=fs)

    axs[2] = fig.add_subplot(313)
    axs[2].set_xlabel('$\sqrt{u^2+v^2}$ (m)',fontsize=fs)
    #axs[2].scatter(radius_C_fields[j],ampl_C_feath[j],s=60,label='ST: visibilities feathered')
    #axs[2].scatter(radius_G_fields[j],ampl_G_feath[j],s=50,label='SA: visibilities feathered')
    axs[2].scatter(data[field]['uvr1d'][band],data[field][stokes]['cgps']['feather'][band],color='blue',label='ST: mean')
    axs[2].scatter(data[field]['uvr1d'][band],data[field][stokes]['gmims']['feather'][band],color='red',label='SA: mean')
    axs[2].set_ylim(0,2000)
    handles,labels = axs[2].get_legend_handles_labels()
    order = [2,0,3,1]
    #axs[2].legend([handles[idx] for idx in order], [labels[idx] for idx in order],fontsize=fs)

    for i in range(0,3):
        axs[i].tick_params(axis="x", labelsize=fs)
        axs[i].tick_params(axis="y", labelsize=fs)
        axs[i].set_xlim(0,50)
        axs[i].set_ylabel('PI (K - uv-plane equiv.)',fontsize=fs)
        #axs[i].plot([12.858,12.858],[0,80000],color='black',linestyle='dashed')
        #axs[i].plot([17.144,17.144,],[0,80000],color='black',linestyle='dashed')
        #axs[i].axvline(x=12.858, color='black',linestyle='dashed')
        axs[i].axvline(x=8.572, color='black',linestyle='dashed')
        axs[i].axvline(x=17.144,color='black',linestyle='dashed')
        axs[i].grid()
        axs[i].set_xticks([0,5,10,15,20,25,30,35,40,45,50])
        #axs[i].set_ticklabels(fontsize=fs)

    #plt.savefig('FIGURES/CGPS_GMIMS_UV_radial_good.pdf')
    
    
    return

## Loop over fields and fill in information

In [ ]:
%%time

#directory = '/home2/DATA/CGPS_GMIMS_PhD/CGPS_GMIMS_PhD/THESIS_pics/'
directory = '/home/ordoga/DRAO_export/CG_W23/UV_IMAGES/'

all_fields = True

counter = 1

for field in fieldnames:
    
    print(counter, field)
    
    if all_fields:
    #if field == 'ob12':
        #uvr_sample = []
    
        for stokes in ['q','u']:

            for chan in ['a','b','c','d']:
                
                try:

                    ######################
                    # GMIMS
                    ######################

                    # GMIMS Raw map
                    file = directory+'f5_G_initial_uv_'+field+'_'+chan+stokes+'.fits'
                    uvmap, hdr = read_data(file)  
                    uvr = get_uv_radii(hdr)

                    if stokes == 'q': # only need to record the radii once
                        data[field]['uvr2d'][chan] = uvr
                        data[field]['uvr1d'][chan] = np.unique(uvr)[0:rmax]

                        #if field == fieldnames[1]:
                        #   uvr_sample.append(uvr)

                    data[field]['stokes'+stokes]['gmims']['raw'][chan] = binning(uvmap,uvr,rmax,data[field]['uvr1d'][chan])

                    # GMIMS deconvolved map
                    file = directory+'f10_G_final_taper_'+field+'_'+chan+stokes+'.fits'
                    uvmap, hdr = read_data(file)
                    data[field]['stokes'+stokes]['gmims']['deconv'][chan] = binning(uvmap,uvr,rmax,data[field]['uvr1d'][chan])

                    # GMIMS feathered map
                    file = directory+'f11_G_feathered_'+field+'_'+chan+stokes+'.fits'
                    uvmap, hdr = read_data(file)
                    data[field]['stokes'+stokes]['gmims']['feather'][chan] = binning(uvmap,uvr,rmax,data[field]['uvr1d'][chan])

                    ######################
                    # CGPS
                    ######################

                    # CGPS raw map
                    file = directory+'f13_C_initial_uv_'+field+'_'+chan+stokes+'.fits'
                    uvmap, hdr = read_data(file)
                    data[field]['stokes'+stokes]['cgps']['raw'][chan] = binning(uvmap,uvr,rmax,data[field]['uvr1d'][chan])

                    # CGPS feathered map
                    file = directory+'f14_C_feathered_'+field+'_'+chan+stokes+'.fits'
                    uvmap, hdr = read_data(file)
                    data[field]['stokes'+stokes]['cgps']['feather'][chan] = binning(uvmap,uvr,rmax,data[field]['uvr1d'][chan])


                    w = np.where((data[field]['uvr1d'][chan]>9) & (data[field]['uvr1d'][chan]<17))
                    C_test = data[field]['stokes'+stokes]['cgps']['raw'][chan][w]
                    G_test = data[field]['stokes'+stokes]['gmims']['deconv'][chan][w]
                    #print(C_test.shape,G_test.shape)
                    data[field]['stokes'+stokes]['ratio'][chan] = np.nanmean(G_test/C_test)
                    
                except:
                    print('no file')
                    pass

            
    counter = counter+1


In [ ]:
print(data['ob12']['stokesq']['ratio']['d'])

np.savez('field_info.npz', data=data)

In [ ]:
make_radial_plots(data,'ob12','stokesu','c')


In [ ]:
print(data['ob03']['uvr']['a'])
print(data['ob03']['uvr']['b'])
print(data['ob03']['uvr']['c'])
print(data['ob03']['uvr']['d'])

In [ ]:
field = 'ob03'
stokes = 'stokesq'
chan = 'a'

#print(data[field][stokes]['gmims']['raw'][chan])
#print(data[field][stokes]['gmims']['feather'][chan])
#print(data[field][stokes]['gmims']['deconv'][chan])
#print('')
#print(data[field][stokes]['cgps']['raw'][chan])
#print(data[field][stokes]['cgps']['feather'][chan])

In [ ]:
this = fits.open('/home2/DATA/CGPS_GMIMS_PhD/CGPS_GMIMS_PhD/THESIS_pics/G_matched_uv_ob03_au.fits')
that = fits.open('/home2/DATA/CGPS_GMIMS_PhD/CGPS_GMIMS_PhD/THESIS_pics/G_feather_uv_ob03_au.fits')

In [ ]:
#plt.imshow(this[0].data[0,0,511-20:511+20,0:20]-that[0].data[0,0,511-20:511+20,0:20],vmin=-0.1,vmax=0.1)
plt.imshow(this[0].data[0,0,511-20:511+20,0:20],vmin=0,vmax=400)

In [ ]:
plt.imshow(that[0].data[0,0,511-20:511+20,0:20],vmin=0,vmax=400)

In [ ]:
C_raw_all_a = []
for field in fieldnames:
    C_raw_all_a.append(data[field]['stokesu']['cgps']['raw']['a'])

In [ ]:
print(C_raw_all_a)

In [ ]:
data['ob12']['stokesq']['cgps']['raw']['a']